# 🔍 Auditoría de Datasets Descomprimidos y Metadatos Reales (GastroAI)

Este cuaderno realiza la inspección directa de las carpetas ya descomprimidas en Google Drive (`/content/drive/MyDrive/GastroAI/dataset/`):
1. **Auditoría por Dataset** (`hyperkvasir`, `egid`, `gastrohun`).
2. **Inspección de Imágenes**: Conteo, dimensiones reales (Ancho x Alto via `PIL.Image`), canales y resoluciones.
3. **Inspección de Metadatos**: Identificación de tablas `.csv`/`.xlsx`, nombres de columnas e identificación de pacientes (`patient_id`).
4. **Estructura de Subcarpetas**: Mapeo de carpetas por hallazgo/clase.
5. **Generación Automática del Reporte Markdown**: Guarda `Reporte_Estructura_Descomprimida_GastroAI.md` en `/GastroAI/reports/`.

In [ ]:
# ==============================================================================
# AUDITORÍA DE CARPETAS DESCOMPRIMIDAS Y METADATOS EN GOOGLE DRIVE
# ==============================================================================
import os
import sys
import glob
import pandas as pd
from PIL import Image
from google.colab import drive

# 1. Montar Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print('✓ Google Drive montado correctamente.')

BASE_DIR = '/content/drive/MyDrive/GastroAI'
DATASET_DIR = os.path.join(BASE_DIR, 'dataset')
REPORTS_DIR = os.path.join(BASE_DIR, 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"Ruta base del dataset: {DATASET_DIR}\n")

md_lines = []
md_lines.append("# Reporte de Auditoría de Datasets Descomprimidos - GastroAI\n")
md_lines.append(f"**Ruta Evaluada**: `{DATASET_DIR}`\n")

# Identificar subdirectorios de datasets
subdirs = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))] if os.path.exists(DATASET_DIR) else []
print(f"Carpetas de datasets detectadas en /dataset/: {subdirs}\n")

for ds_name in sorted(subdirs):
    ds_path = os.path.join(DATASET_DIR, ds_name)
    print(f"====================================================")
    print(f" 📁 INSPECCIONANDO DATASET: {ds_name.upper()}")
    print(f"====================================================")
    
    md_lines.append(f"## Dataset: `{ds_name}`\n")
    md_lines.append(f"**Ruta**: `{ds_path}`\n")
    
    # 1. Buscar archivos de metadatos (.csv, .xlsx, .json)
    meta_files = []
    for root, _, files in os.walk(ds_path):
        for f in files:
            if f.lower().endswith(('.csv', '.xlsx', '.xls', '.json')):
                meta_files.append(os.path.join(root, f))
                
    md_lines.append("### 1. Archivos de Metadatos Encontrados\n")
    if meta_files:
        for mf in meta_files:
            rel_mf = os.path.relpath(mf, ds_path)
            print(f" 📄 Metadato detectado: {rel_mf}")
            md_lines.append(f"- **Archivo**: `{rel_mf}`")
            try:
                if mf.endswith(('.xlsx', '.xls')):
                    df_m = pd.read_excel(mf)
                elif mf.endswith('.csv'):
                    df_m = pd.read_csv(mf)
                else:
                    df_m = pd.read_json(mf)
                    
                md_lines.append(f"  - **Registros (filas)**: {len(df_m)}")
                md_lines.append(f"  - **Columnas ({len(df_m.columns)})**: `{list(df_m.columns)}`")
                md_lines.append("  - **Muestra (3 filas)**:\n")
                md_lines.append(df_m.head(3).to_markdown(index=False))
                md_lines.append("\n")
                
                print(f"   • Filas: {len(df_m)} | Columnas: {list(df_m.columns)[:5]}")
            except Exception as e:
                print(f"   ❌ Error leyendo {rel_mf}: {e}")
                md_lines.append(f"  - ❌ *Error al leer*: {e}\n")
    else:
        print(" ⚠️ No se encontraron archivos de metadatos en esta carpeta.")
        md_lines.append("*No contiene tablas explícitas de metadatos (.csv / .xlsx).*\n")
        
    # 2. Escanear imágenes
    img_paths = []
    subfolder_counts = {}
    for root, _, files in os.walk(ds_path):
        rel_root = os.path.relpath(root, ds_path)
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif')):
                full_img_p = os.path.join(root, f)
                img_paths.append(full_img_p)
                subfolder_counts[rel_root] = subfolder_counts.get(rel_root, 0) + 1
                
    total_imgs = len(img_paths)
    print(f"\n 🖼️ Total de imágenes encontradas: {total_imgs}")
    md_lines.append(f"### 2. Análisis de Imágenes ({total_imgs} imágenes totales)\n")
    
    if total_imgs > 0:
        sample_size = min(total_imgs, 100)
        sample_imgs = img_paths[:sample_size]
        
        widths, heights, modes = [], [], []
        for imp in sample_imgs:
            try:
                with Image.open(imp) as im:
                    w, h = im.size
                    widths.append(w)
                    heights.append(h)
                    modes.append(im.mode)
            except Exception:
                pass
                
        if widths:
            min_w, max_w, avg_w = min(widths), max(widths), int(sum(widths)/len(widths))
            min_h, max_h, avg_h = min(heights), max(heights), int(sum(heights)/len(heights))
            common_mode = pd.Series(modes).mode()[0] if modes else 'Desconocido'
            
            print(f"   • Resoluciones (Muestra de {len(widths)}): Ancho min/prom/max = {min_w}/{avg_w}/{max_w} | Alto min/prom/max = {min_h}/{avg_h}/{max_h}")
            print(f"   • Formato/Modo cromático predominante: {common_mode}")
            
            md_lines.append("**Propiedades Físicas de las Imágenes (Mapeo de Muestra):**")
            md_lines.append(f"- **Dimensiones de Ancho (px)**: Mínimo = `{min_w}px`, Promedio = `{avg_w}px`, Máximo = `{max_w}px`")
            md_lines.append(f"- **Dimensiones de Alto (px)**: Mínimo = `{min_h}px`, Promedio = `{avg_h}px`, Máximo = `{max_h}px`")
            md_lines.append(f"- **Modo de Color Predominante**: `{common_mode}`")
            md_lines.append("\n")
            
        md_lines.append("**Estructura de Subcarpetas / Distribución en Disco:**\n")
        md_lines.append("| Subcarpeta / Ruta Relativa | Conteo de Imágenes |")
        md_lines.append("| :--- | :--- |")
        print("   • Distribución por subcarpetas (Top 10):")
        for folder_rel, count in list(subfolder_counts.items())[:15]:
            print(f"     - `{folder_rel}`: {count} imágenes")
            md_lines.append(f"| `{folder_rel}` | {count} |")
        md_lines.append("\n")
    else:
        md_lines.append("*No se hallaron imágenes descompresas en este directorio.*\n")
        
    md_lines.append("---\n")

out_md_path = os.path.join(REPORTS_DIR, 'Reporte_Estructura_Descomprimida_GastroAI.md')
with open(out_md_path, 'w', encoding='utf-8') as f:
    f.write("\n".join(md_lines))

print("====================================================")
print("✓ INFORME DE DIAGNÓSTICO DESCOMPRIMIDO GENERADO")
print(f"📍 Guardado físicamente en: {out_md_path}")
print("====================================================")
